## **Sección 2**: *Tabu Search*

### 2.1 Implementación del algoritmo

> Implemente el algoritmo de *Tabu Search* utilizando **únicamente la lista tabú básica con tenure fijo**, sin estrategias avanzadas de intensificación/diversificación. Emplee como criterio de parada un número máximo de iteraciones $N_{\text{max}}$, y considere como criterio complementario el **Estancamiento**, deteniendo la búsqueda si no hay mejora en el mejor fitness durante $k$ iteraciones consecutivas. Puede encontrar más detalles del algoritmo en el **Anexo II**.

### 2.2 Análisis de hiperparámetros en Schwefel 1D

Los dos hiperparámetros fundamentales de Tabu Search básico son el tamaño de la lista tabú (tenure) $k$ y el tamaño del vecindario $|N|$. El tenure controla el balance entre intensificación (valores pequeños permiten revisitar regiones rápidamente) y diversificación (valores grandes fuerzan exploración prolongada de nuevas regiones). El tamaño del vecindario determina qué tan exhaustivamente se explora el entorno local en cada iteración, afectando directamente el costo computacional.

#### Experimento 1: Tamaño de lista tabú (tenure)

**Configuración experimental**:

$$
\begin{aligned}
\text{Función:} & \quad f_1(x) = -x \cdot \sin(\sqrt{|x|}), \quad x \in [-512, 512] \\
\text{Hiperparámetros:} & \quad k \in \{5, 10, 20, 50, 100\} \\
& \quad |N| = 20 \text{ (fijo)} \\
\text{Iteraciones:} & \quad N_{\text{max}} = 1000 \\
\text{Repeticiones:} & \quad 10 \text{ ejecuciones independientes por cada } k \\
\text{Inicialización:} & \quad \text{Aleatoria uniforme en } [-512, 512] \\
\text{Generación de vecinos:} & \quad x_i = x + \mathcal{N}(0, \sigma^2), \quad i=1,\ldots,20, \quad \sigma = 50
\end{aligned}
$$

**Justificación del rango de tenure** (tamaño de la memoria): 
- $k = 5$: memoria muy corta, poco efecto de diversificación, comportamiento cercano a Hill Climbing
- $k = 20$: valor intermedio típico en literatura
- $k = 100$: memoria larga, fuerte diversificación, riesgo de prohibir regiones prometedoras demasiado tiempo

**Métricas registradas** (por cada repetición):
- Mejor fitness alcanzado
- Posición de mejor solución
- Iteración del mejor hallazgo
- Número total de vecinos evaluados
- Proporción promedio de vecinos que fueron tabú

> **Análisis de los resultados**: Idéntico al protocolo de SA: media, mediana, min, max, desviación estándar, organizados en tabla comparativa. Generar boxplots y gráficos de convergencia promedio.


**Preguntas de análisis**:

1. ¿Existe un valor óptimo de $k$ o el rendimiento es insensible al tenure en este rango?
2. ¿Tenure muy pequeño ($k=5$) resulta en estancamiento por falta de diversificación?
3. ¿Tenure muy grande ($k=100$) perjudica por prohibir regiones prometedoras?
4. ¿Cómo se relaciona la variabilidad de resultados con el tamaño del tenure?
5. ¿En qué momento de la ejecución típicamente se encuentra el mejor resultado para cada $k$?


#### Experimento 2: Tamaño de vecindario

> Emplee el valor de tenure identificado previamente (mantener fijo) y evalúe el impacto del tamaño del vecindario explorado en cada iteración.

**Configuración experimental**:

$$
\begin{aligned}
\text{Hiperparámetros:} & \quad k = [k_{\text{óptimo}} \text{ del Exp.1}] \text{ (fijo)} \\
& \quad |N| \in \{5, 10, 20, 50, 100\} \\
\text{Protocolo:} & \quad \text{Idéntico al Experimento 1}
\end{aligned}
$$

**Observación importante sobre el número de evaluaciones**: El número total de evaluaciones de función objetivo varía con $|N|$:
- $|N| = 5$: $1000 \times 5 = 5000$ evaluaciones totales
- $|N| = 20$: $1000 \times 20 = 20000$ evaluaciones totales
- $|N| = 100$: $1000 \times 100 = 100000$ evaluaciones totales

Esto introduce un sesgo: configuraciones con $|N|$ mayor tienen más evaluaciones totales. Para una comparación completamente justa, se debería fijar el número total de evaluaciones y ajustar el número de iteraciones correspondientemente. Sin embargo, para este experimento preliminar se mantiene fijo el número de iteraciones para observar el trade-off entre amplitud de exploración local versus número de iteraciones.


**Preguntas de análisis**:

1. ¿Mayor $|N|$ siempre resulta en mejor rendimiento, o existe un punto de rendimientos decrecientes?
2. ¿Cómo afecta $|N|$ al tiempo de ejecución?
3. ¿El $|N|$ óptimo depende del valor de $k$ elegido?
4. ¿Explorar vecindarios muy grandes ($|N|=100$) en cada iteración es computacionalmente justificable?


### 2.3 Aplicación a la función 2D

Los experimentos previos en la función unidimensional de Schwefel han permitido identificar configuraciones óptimas de los hiperparámetros $k$ (tenure) y $|N|$ (tamaño de vecindario) para *Tabu Search*. En esta sección se aplicará el algoritmo a la función $f_2(x,y)$, adaptando la implementación al espacio bidimensional y evaluando el impacto de los hiperparámetros en un paisaje de fitness con estructura radial y alta multimodalidad.

#### Adaptaciones necesarias para 2D

**Operador de vecindario bidimensional**: La generación de vecinos en $\mathbb{R}^2$ se realiza mediante perturbación Gaussiana independiente en cada dimensión. Para cada iteración, se generan $|N|$ vecinos candidatos:

$$\begin{pmatrix} x'_i \\ y'_i \end{pmatrix} = \begin{pmatrix} x \\ y \end{pmatrix} + \begin{pmatrix} \delta_{x,i} \\ \delta_{y,i} \end{pmatrix}, \quad \delta_{x,i}, \delta_{y,i} \sim \mathcal{N}(0, \sigma^2), \quad i = 1, \ldots, |N|$$

donde $\sigma$ es la desviación estándar que controla el tamaño del paso de búsqueda. Un valor apropiado inicial para la función $f_2(x,y)$ con dominio $[-100, 100]^2$, es $\sigma = 10.0$, consistente con el valor empleado en *Simulated Annealing*. Este valor permite exploración efectiva del espacio sin generar pasos excesivamente grandes que dificulten la convergencia.

**Manejo de restricciones del dominio**: Cuando la perturbación genera un candidato $(x'_i, y'_i)$ que viola las restricciones del dominio, se aplica la estrategia de **proyección al borde**: si $x'_i < -100$ se fija $x'_i = -100$, si $x'_i > 100$ se fija $x'_i = 100$, y análogamente para $y'_i$. Esto garantiza que todos los candidatos evaluados sean factibles.

**Representación de la lista tabú en 2D**: La lista tabú almacena puntos $(x, y)$ recientemente visitados. Dado que trabajamos con números reales de punto flotante, dos soluciones se consideran equivalentes si su distancia euclidiana es menor que una tolerancia $\varepsilon$:

$$\text{is\_tabu}((x', y')) = \exists (x_t, y_t) \in \text{TabuList} : \sqrt{(x' - x_t)^2 + (y' - y_t)^2} < \varepsilon$$

Para la función $f_2(x,y)$, se empleará $\varepsilon = 1.0$ como criterio de proximidad, representando aproximadamente el 1% del rango del dominio.


**Visualización en 2D**:

- **Gráfico de contorno** (contour plot) de $f_2(x,y)$ en el dominio $[-100,100]^2$ como referencia estática de fondo
- **Soluciones tabú**: marcadores semi-transparentes en color mostrando las posiciones actualmente en la lista tabú, permitiendo visualizar qué regiones están temporalmente prohibidas
- **Posición actual**: marcador de color que muestra la ubicación de la solución actual $(x_{\text{current}}, y_{\text{current}})$
- **Mejor posición**: marcador de color en forma de estrella que indica $(x_{\text{best}}, y_{\text{best}})$
- **Óptimo global conocido**: marcador negro en el origen $(0, 0)$ para referencia

Esta visualización permite observar cómo la memoria explícita de Tabu Search moldea la trayectoria de búsqueda en el paisaje radial, y evaluar si el algoritmo logra aproximarse sistemáticamente al óptimo global.

#### Experimento 1: Calibración del tamaño de lista tabú (tenure) en 2D

El primer paso es determinar si el tenure óptimo identificado en Schwefel 1D es apropiado para la función de Oscilación Radial, considerando que el espacio de búsqueda bidimensional presenta mayor complejidad y un número mayor de regiones locales que pueden ser visitadas.

**Configuración experimental**:

$$
\begin{aligned}
\text{Función:} & \quad f_2(x,y) = (x^{2} + y^{2})^{0.25} \cdot [\sin^{2}(50 \cdot (x^{2} + y^{2})^{0.1}) + 1] \\
\text{Dominio:} & \quad (x,y) \in [-100, 100]^2 \\
\text{Hiperparámetros:} & \quad k \in \{5, 10, 20, 50, 100\} \\
& \quad |N| = [|N|_{\text{óptimo}} \text{ de Exp. 2.3.2}] \text{ (fijo)} \\
& \quad \sigma = 10.0 \text{ (desviación estándar de perturbación)} \\
& \quad \varepsilon = 1.0 \text{ (tolerancia para comparación tabú)} \\
\text{Iteraciones:} & \quad N_{\text{max}} = 2000 \\
\text{Repeticiones:} & \quad 10 \text{ ejecuciones independientes por cada } k \\
\text{Inicialización:} & \quad (x_0, y_0) \text{ aleatorio uniforme en } [-100, 100]^2
\end{aligned}
$$

**Justificación del rango de tenure**:
- $k = 5$: memoria muy corta, permite revisitar regiones rápidamente, riesgo de ciclos
- $k = 20$: memoria intermedia, balance típico en literatura
- $k = 50$: memoria larga, fuerte diversificación
- $k = 100$: memoria muy larga, riesgo de excesiva prohibición de regiones prometedoras

En 2D, el espacio de búsqueda es considerablemente más amplio que en 1D, lo que podría requerir tenures más largos para evitar ciclos efectivamente. Sin embargo, tenures excesivos pueden prohibir prematuramente regiones cercanas al óptimo global.


**Métricas a registrar** (para cada una de las 10 repeticiones de cada configuración):

1. **Mejor fitness alcanzado**: $f_2(x_{\text{best}}, y_{\text{best}})$
2. **Posición de la mejor solución**: $(x_{\text{best}}, y_{\text{best}})$
3. **Distancia euclidiana al óptimo global**: $d = \sqrt{x_{\text{best}}^2 + y_{\text{best}}^2}$
4. **Iteración en que se encontró el mejor**: $t_{\text{best}}$
6. **Número total de evaluaciones**: $N_{\text{max}} \times |N|$ (debería ser constante)
7. **Proporción promedio de vecinos tabú**: fracción del vecindario generado que fue rechazado por estar en la lista tabú

> Para cada valor de $k$, calcular sobre las 10 repeticiones:
> - Media aritmética de $f_{\text{best}}$
> - Mediana, mínimo y máximo de $f_{\text{best}}$
> - Desviación estándar
> - Media de la distancia al óptimo $\bar{d}$
> - Proporción promedio de vecinos tabú
> Presentar los resultados en una **tabla comparativa**:



| $k$ | Media | Mediana | Mín | Máx | Dist. promedio | % Tabú |
|-----|:-----:|---------|-----|-----|----------------|--------|
| 5   | ...   | ...     | ... | ... | ...            | ...    |
| 10  | ...   | ...     | ... | ... | ...            | ...    |
| 20  | ...   | ...     | ... | ... | ...            | ...    |
| 50  | ...   | ...     | ... | ... | ...            | ...    |
| 100 | ...   | ...     | ... | ... | ...            | ...    |



> Adicionalmente, generar visualizaciones complementarias:
> 1. **Boxplot comparativo**: Un gráfico de boxplot que muestre, para cada valor de $k$, la distribución de los mejores fitness alcanzados.
> 2. **Scatter plot de posiciones finales**: Sobre un contour plot de la función, marcar las 10 posiciones finales $(x_{\text{best}}, y_{\text{best}})$ de cada configuración con diferentes colores. Incluir un círculo en el origen y círculos concéntricos a distancias 10, 25, 50 como referencia visual. Opcionalmente, puede utilizar una proyección superior de la función $f_2(x,y)$ para ubicar los puntos.
> 3. **Gráfico de convergencia promedio**: Para cada $k$, graficar la evolución del fitness promedio (promediado sobre las 10 repeticiones) vs iteración.
> 4. **Proporción de vecinos tabú vs tenure**: Gráfico de barras mostrando el porcentaje promedio de vecinos rechazados para cada valor de $k$, permitiendo cuantificar el efecto de diversificación.


**Preguntas de análisis**:

1. ¿Qué valor de $k$ produce el mejor rendimiento promedio en términos de fitness alcanzado?
2. ¿El tenure óptimo en 2D coincide con el identificado en Schwefel 1D, o requiere ajuste? ¿El espacio 2D requiere memoria más larga?
3. ¿Cómo se relaciona la proporción de vecinos tabú con la calidad de las soluciones finales? ¿Mayor prohibición mejora o perjudica el rendimiento?
4. ¿Tenures muy pequeños ($k = 5$) resultan en ciclos observables en las visualizaciones? ¿Tenures muy grandes ($k = 100$) impiden acercarse al óptimo?
5. ¿Cuántas de las 10 repeticiones de cada configuración logran encontrar soluciones dentro de un radio $d < 10$ del óptimo?

#### Experimento 2: Calibración del tamaño de vecindario en 2D

El tamaño del vecindario $|N|$ determina cuán exhaustivamente se explora el entorno local de la solución actual en cada iteración. En problemas 2D, un vecindario más amplio puede ser necesario para capturar adecuadamente la estructura del paisaje, pero incrementa proporcionalmente el costo computacional por iteración.


**Configuración experimental**:

$$
\begin{aligned}
\text{Hiperparámetros:} & \quad k = [k_{\text{óptimo}} \text{ del Exp. 1}] \text{ (fijo)} \\
& \quad |N| \in \{10, 20, 30, 50, 100\} \\
& \quad \sigma = 10.0 \text{ (fijo)} \\
& \quad \varepsilon = 1.0 \text{ (fijo)} \\
\text{Protocolo:} & \quad \text{Idéntico al Experimento 1}
\end{aligned}
$$


**Justificación del rango de $|N|$**:
- $|N| = 10$: exploración local limitada, pocas opciones por iteración
- $|N| = 30$: exploración moderada, balance razonable
- $|N| = 100$: exploración exhaustiva del entorno local, alto costo computacional

**Observación crítica sobre presupuesto computacional**: A diferencia del Experimento 1 donde el número total de evaluaciones era constante, aquí varía con $|N|$:

- $|N| = 10$: $2000 \times 10 = 20{,}000$ evaluaciones totales
- $|N| = 50$: $2000 \times 50 = 100{,}000$ evaluaciones totales
- $|N| = 100$: $2000 \times 100 = 200{,}000$ evaluaciones totales

Esta variación introduce un sesgo: configuraciones con mayor $|N|$ tienen más evaluaciones disponibles. Sin embargo, este experimento busca evaluar el trade-off entre amplitud de exploración local versus número de iteraciones, por lo que se mantiene fijo $N_{\text{max}}$ en lugar del presupuesto total.


> Aplicar el mismo protocolo estadístico del Experimento 1, prestando especial atención a:
> - **Relación costo-beneficio**: ¿El incremento en calidad de soluciones justifica el incremento en evaluaciones?
> - **Tiempo de ejecución**: ¿Cómo escala el tiempo con $|N|$?
> - **Proporción de vecinos tabú**: ¿Vecindarios grandes resultan en mayor o menor proporción de candidatos prohibidos?
> Adicionalmente, para cada valor de $|N|$, calcular:
> - **Número promedio de mejoras por iteración**: frecuencia con que se actualiza $f_{\text{best}}$
> - **Eficiencia computacional**: calidad de solución alcanzada por cada 10000 evaluaciones (puede construir un gráfico de barras mostrando los valores calculados, comparando para los distintos $|N|$ analizados)


**Preguntas de análisis**:
1. ¿Qué valor de $|N|$ produce el mejor balance entre calidad de soluciones y costo computacional?
2. ¿Existe un punto de rendimientos decrecientes donde incrementar $|N|$ no mejora sustancialmente los resultados?
3. ¿Cómo interactúa $|N|$ con el tenure $k$? ¿Vecindarios grandes requieren tenures diferentes?
4. Observando el tiempo de ejecución, ¿es práctico usar $|N| = 100$ considerando el beneficio marginal obtenido?
5. Comparando con el Experimento 2.3.2 en Schwefel 1D, ¿el tamaño de vecindario óptimo es diferente en 2D?

#### Experimento 3: Impacto de la tolerancia de proximidad ($\varepsilon$)

El parámetro $\varepsilon$ determina cuán estricta es la comparación de proximidad al verificar si un candidato está en la lista tabú. Un $\varepsilon$ muy pequeño puede permitir que soluciones muy cercanas (pero no idénticas) sean consideradas diferentes, potencialmente causando ciclos. Un $\varepsilon$ muy grande puede prohibir excesivamente regiones amplias alrededor de cada punto tabú.


**Configuración experimental**:

$$
\begin{aligned}
\text{Hiperparámetros:} & \quad k = [k_{\text{óptimo}} \text{ del Exp. 1}] \text{ (fijo)} \\
& \quad |N| = [|N|_{\text{óptimo}} \text{ del Exp. 2}] \text{ (fijo)} \\
& \quad \sigma = 10.0 \text{ (fijo)} \\
& \quad \varepsilon \in \{0.1, 0.5, 1.0, 2.0, 5.0\} \\
\text{Protocolo:} & \quad \text{Idéntico al Experimento 1}
\end{aligned}
$$



**Justificación del rango de $\varepsilon$**:
- $\varepsilon = 0.1$: tolerancia muy estricta, solo puntos casi idénticos son considerados tabú
- $\varepsilon = 1.0$: tolerancia moderada (~1% del dominio)
- $\varepsilon = 5.0$: tolerancia amplia (~5% del dominio), prohibición de regiones grandes


> Aplicar el mismo protocolo estadístico que el los experimentos anteriores.


**Preguntas de análisis**:

1. ¿Qué valor de $\varepsilon$ produce el mejor equilibrio entre evitar ciclos y permitir refinamiento local?
2. ¿Tolerancias muy estrictas ($\varepsilon = 0.1$) resultan en comportamiento cualitativamente similar a no tener lista tabú?
3. ¿Tolerancias muy amplias ($\varepsilon = 5.0$) impiden convergencia fina al óptimo por prohibición excesiva?
4. ¿El efecto de $\varepsilon$ es más pronunciado con tenures grandes o pequeños?
5. Observando las visualizaciones, ¿cómo afecta $\varepsilon$ a los patrones espaciales de la búsqueda?


#### Análisis integrador

Una vez completados los tres experimentos, se debe realizar una síntesis que responda:

1. **Transferencia de hiperparámetros**: ¿Los valores óptimos de $k$ y $|N|$ identificados en Schwefel 1D transfieren efectivamente a la función $f_2(x,y)$ 2D, o requieren ajuste sustancial? ¿La dimensionalidad afecta más al tenure o al tamaño de vecindario?

2. **Rol de la tolerancia de proximidad**: ¿El parámetro $\varepsilon$ tiene un impacto comparable a $k$ y $|N|$ en el rendimiento, o su efecto es secundario? ¿Es posible identificar un valor $\varepsilon$ "universal" apropiado, o debe ajustarse según la función?

3. **Desafíos de dimensionalidad**: ¿La función 2D presenta desafíos cualitativamente diferentes a la 1D para Tabu Search? Por ejemplo, ¿la memoria tabú es más o menos efectiva en 2D para evitar ciclos? ¿La proporción de vecinos tabú es mayor o menor?

4. **Visualización del comportamiento**: Basándose en las visualizaciones construidas durante los experimentos, describir cualitativamente cómo Tabu Search navega el paisaje radial con oscilaciones. ¿Las prohibiciones tabú crean patrones espaciales reconocibles (ej: trayectorias que evitan sistemáticamente ciertas regiones)? ¿El algoritmo exhibe comportamiento exploratorio continuo o tiende a estabilizarse en regiones específicas?

5. **Comparación con Simulated Annealing**: Contrastar cualitativamente el comportamiento de Tabu Search en 2D (basándose en visualizaciones y métricas) con el comportamiento de Simulated Annealing (Sección 1.4). ¿Cuál algoritmo parece navegar más eficientemente el paisaje de la función $f_2(x,y)$? ¿Las trayectorias de búsqueda son cualitativamente diferentes?



---